# Experiment B: Advanced Feature Engineering

이 노트북은 EDA 결과를 바탕으로 **시계열 특성(Rolling Stats 및 Lag)** 및 **Target Encoding**을 추가하여 데이터 자체의 질과 정보량을 극대화하는 실험입니다.

## 1. 환경 설정 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from category_encoders import TargetEncoder
import gc

DATA_PATH = '/kaggle/input/competitions/playground-series-s6e5/'
train = pd.read_csv(DATA_PATH + 'train.csv')
test = pd.read_csv(DATA_PATH + 'test.csv')
submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')

print(f'Train Shape: {train.shape}, Test Shape: {test.shape}')

## 2. 시계열 및 물리적 특성 공학 (Rolling & Lag Features)
기존 EDA 인사이트에 더해, 각 드라이버의 직전 랩 변화량(Lag)과 최근 3랩의 랩타임 변화 추세(Rolling Mean)를 포착하여 타이어 성능 저하(Cliff) 시점을 세밀하게 예측합니다.

In [ ]:
def advanced_engineering(df):
    # 1. 기존 EDA 기반 특성
    compound_mean_life = {'HARD': 25, 'MEDIUM': 18, 'SOFT': 12, 'INTERMEDIATE': 20, 'WET': 15}
    df['Relative_TyreLife'] = df['TyreLife'] / df['Compound'].map(compound_mean_life).fillna(20)
    df['Is_Final_Laps'] = (df['RaceProgress'] > 0.85).astype(int)
    df['Degradation_Momentum'] = df['TyreLife'] * df['Cumulative_Degradation']
    
    # 2. 시계열적 흐름 특성 (Rolling & Lag)
    # 시간순 정렬 (Race -> Driver -> LapNumber)
    df = df.sort_values(by=['Race', 'Driver', 'LapNumber'])
    
    # 1~2랩 전의 랩타임 변화량 (Lag)
    df['LapTime_Delta_Lag1'] = df.groupby(['Race', 'Driver'])['LapTime_Delta'].shift(1).fillna(0)
    df['LapTime_Delta_Lag2'] = df.groupby(['Race', 'Driver'])['LapTime_Delta'].shift(2).fillna(0)
    
    # 최근 3랩 평균 랩타임 변화량 (추세)
    df['Rolling_Mean_Delta_3'] = df.groupby(['Race', 'Driver'])['LapTime_Delta'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    
    # 원래 인덱스로 복구
    return df.sort_index()

train = advanced_engineering(train)
test = advanced_engineering(test)
print("Advanced Feature Engineering Complete.")

## 3. Target Encoding 및 모델 검증 전략 세팅
`Driver` 변수에 대해 Target Encoding을 적용합니다. 
※ 주의: 정보 누수(Data Leakage)와 과적합을 막기 위해 반드시 교차 검증(K-Fold) 루프 안에서 Train Fold로만 fit을 수행해야 합니다.

In [ ]:
cat_features = ['Driver', 'Compound', 'Race', 'Year']
drop_cols = ['id', 'PitNextLap']
features = [c for c in train.columns if c not in drop_cols]

X = train[features]
y = train['PitNextLap']
groups = train['Race']

kf = GroupKFold(n_splits=5)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

## 4. 모델 학습 및 Target Encoding 동시 진행
각 Fold별로 데이터가 나누어지면, 해당 훈련 데이터만을 사용해 `Driver`별 평균 피트인 확률을 계산(Target Encoding)한 뒤 모델을 학습합니다.

In [ ]:
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y, groups)):
    print(f"--- Fold {fold+1} ---")
    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    X_test = test[features].copy()
    
    # Target Encoding 적용 (Driver 기준, Smoothing을 통해 극단값 보정)
    encoder = TargetEncoder(cols=['Driver'], smoothing=10)
    X_train['Driver_TE'] = encoder.fit_transform(X_train['Driver'], y_train)
    X_val['Driver_TE'] = encoder.transform(X_val['Driver'])
    X_test['Driver_TE'] = encoder.transform(X_test['Driver'])
    
    # CatBoost 학습
    model = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, eval_metric='AUC', verbose=100, random_seed=42)
    model.fit(X_train, y_train, cat_features=cat_features, eval_set=(X_val, y_val), early_stopping_rounds=50)
    
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / 5
    
    print(f'Fold {fold+1} Validation AUC: {roc_auc_score(y_val, oof_preds[val_idx]):.4f}\n')

print(f'==> Overall OOF AUC: {roc_auc_score(y, oof_preds):.4f}')

## 5. 최종 제출 파일 생성

In [ ]:
submission['PitNextLap'] = test_preds
submission.to_csv('submission_exp_B.csv', index=False)
print("\nSubmission file 'submission_exp_B.csv' saved successfully.")